# 4. 🟪 출력 — 배준호

<small>마지막 수정: 2026-11-24 19:21 (한국시간) · v16</small>


## 🟪 출력 흐름

팀 입력·RAG → 대화·프롬프트 → Qwen 답변 → 검사·선택지 → 게임 화면

**복붙용 출력 셀입니다.** `BAE` 폴더를 GitHub `TRPG-masterbot/data`의 `main`에 올린 뒤, 최신 통합 v1.3의 출력 부분만 교체합니다.


## 🟪 O1. 파일·모델 설정

GitHub의 BAE 보조 파일을 확인하고 공통 모델 설정을 연결합니다. 같은 파일은 다시 받지 않습니다.


In [ ]:
# 마지막 수정: 2026-11-24 19:21 (한국시간) · v16
"""팀 입력 JSON·원본 RAG 결과를 Ollama의 한국어 대사와 장면으로 연결합니다.

출력 파트는 검색기를 다시 만들거나 인물 지식을 추가하지 않습니다.
게임 상태 변경은 엔진이 담당하고 여기서는 전달된 결과를 표현합니다.
"""

# GitHub의 보조 코드만 연결합니다. 대화 생성의 핵심은 이 셀 아래와 O2~O5에 있습니다.
import hashlib as _bae_hashlib
import importlib.util as _bae_importlib
from pathlib import Path as _BaePath
import sys as _bae_sys
from urllib.request import urlopen as _bae_urlopen

BAE_TEAM_BINDINGS = {k: globals()[k] for k in ("json", "re", "time") if k in globals()}
BAE_FILES_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/BAE/src/"
BAE_SETUP_SHA256 = "f69cc3efcdc58c4676545235935f5fa16aa20babffd26f11809a989dc59aec11"
BAE_MANIFEST_SHA256 = "0303acd52b3fe3018a9f852e99e54f7cb84501cd0bf2e2133a8e62e3da1c5dc3"
_bae_base = _BaePath("/content") if "google.colab" in _bae_sys.modules else _BaePath.cwd()
_bae_cache = _bae_base / ".bae_cache" / BAE_SETUP_SHA256 / "runtime_setup.py"
_bae_candidates = (_bae_base / "BAE/src/runtime_setup.py", _bae_base / "src/runtime_setup.py", _bae_cache)
_bae_setup = next((p for p in _bae_candidates if p.is_file() and _bae_hashlib.sha256(p.read_bytes()).hexdigest() == BAE_SETUP_SHA256), None)
if _bae_setup is None:
    with _bae_urlopen(BAE_FILES_URL + "runtime_setup.py", timeout=60) as _bae_response:
        _bae_raw = _bae_response.read()
    if _bae_hashlib.sha256(_bae_raw).hexdigest() != BAE_SETUP_SHA256:
        raise ValueError("GitHub의 BAE 파일 버전이 다릅니다. 같은 버전의 BAE 폴더를 먼저 업로드하세요.")
    _bae_cache.parent.mkdir(parents=True, exist_ok=True)
    _bae_cache.write_bytes(_bae_raw)
    _bae_setup = _bae_cache
_bae_spec = _bae_importlib.spec_from_file_location("_bae_runtime_setup_" + BAE_SETUP_SHA256[:12], _bae_setup)
_BAE_SETUP_MODULE = _bae_importlib.module_from_spec(_bae_spec)
_bae_spec.loader.exec_module(_BAE_SETUP_MODULE)
BAE_RUNTIME_PATH, BAE_RUNTIME_PREPARE_INFO = _BAE_SETUP_MODULE.prepare_runtime(
    BAE_FILES_URL, BAE_MANIFEST_SHA256, target=globals().get("_BAE_HELPER_TEST_PATH"),
    previous_root=globals().get("BAE_RUNTIME_PATH"))


def _bae_load_output_support(name):
    """반복 처리 보조 함수도 원래 출력 셀과 같은 공간에서 정의합니다."""
    _BAE_SETUP_MODULE.load_support(globals(), name.removesuffix(".py"))


# 이전 출력 버전의 파일 준비 함수가 진단 기록에 섞이지 않게 정리합니다.
for _bae_old_name in ("_bae_prepare_direct_helpers", "_bae_capture_team_bindings"):
    globals().pop(_bae_old_name, None)

# O1. 공통 모델 설정을 출력에 연결하고 실행 제어 도구를 준비합니다.
# 이 셀 앞부분의 GitHub 파일 준비는 최종 배포 단계에서 붙입니다.
import json
import hashlib
import re
import secrets
import threading
import time
from contextlib import contextmanager
from contextvars import ContextVar
from copy import deepcopy
_SOURCE_SHA256 = None
OUTPUT_VERSION = "output-v16-20261124"
MODEL_NAME = GEN_MODEL
OLLAMA_BASE_URL = OLLAMA_URL
OLLAMA_KEEP_ALIVE = "30m"
OLLAMA_NUM_CTX = NUM_CTX
MAX_NEW_TOKENS = 384          # 한 번의 본문 생성 한도. 잘린 응답은 그대로 표시하지 않습니다.
MAX_INPUT_CHARACTERS = 24000  # 요청이 지나치게 커지면 오래된 대화부터 줄입니다.
HISTORY_MESSAGES = 12         # 현재 NPC와 나눈 최근 발언을 가져오는 최대 개수입니다.
SUGGESTION_COUNT = 2          # 생성 추천의 기본 개수. 실제 호출에서 조정할 수 있습니다.
DO_SAMPLE = False            # 임의 추출을 끄고 같은 설정의 답변 변동을 줄입니다.
TEMPERATURE, TOP_P, TOP_K = .7, .8, 20
output_model, input_namespace = None, None
_OUTPUT_RUNTIME_INFO = {"runtime_package": deepcopy(globals().get("BAE_RUNTIME_PREPARE_INFO", {}))}     # 실제 GPU·모델 연결 상태를 복사 기록에 함께 남깁니다.
generation_lock = threading.RLock()  # 동시에 여러 생성 요청이 GPU를 점유하지 않게 합니다.
_output_stream_sink = ContextVar("trpg_output_stream_sink", default=None)  # 현재 요청의 화면 콜백
_output_stream_cancel_events = ContextVar("trpg_output_stream_cancel_events", default=())

_bae_load_output_support("execution_control.py")


## 🟪 O2. 팀 입력 JSON 연결

팀 I2·I3를 그대로 호출해 질문·게임 상태·검색 근거를 전달받습니다.


In [ ]:
# O2. 팀 I2 검색과 I3 JSON을 그대로 연결합니다.
# 실제 발언과 검색용 문장을 구분하고, 원본 검색 결과를 출력 자료로 받습니다.
_bae_load_output_support("input_helpers.py")

def bind_team_backend(search_rag, handoff_builder, scenario_cards=None, can_search=None,
                      *, settings=None, gen_model=None, ollama_url=None, num_ctx=None):
    """준비된 팀 I2·I3 함수와 공통 설정을 연결합니다. 여기서는 검색·모델 준비를 실행하지 않습니다.

    settings=globals()로 전달하면 GEN_MODEL·OLLAMA_URL·NUM_CTX만 읽습니다.
    scenario_cards와 can_search는 UI 연결용 원본 참조이며 지식 보충에 쓰지 않습니다.
    """
    global input_namespace, MODEL_NAME, OLLAMA_BASE_URL, OLLAMA_NUM_CTX
    if not callable(search_rag) or not callable(handoff_builder):
        raise TypeError("팀 검색 함수와 handoff 생성 함수가 필요합니다.")
    if can_search is not None and not callable(can_search):
        raise TypeError("can_search는 원본 공개 조건 함수여야 합니다.")
    config = settings or {}
    model = str(gen_model or config.get("GEN_MODEL", MODEL_NAME)).strip()
    url = str(ollama_url or config.get("OLLAMA_URL", OLLAMA_BASE_URL)).rstrip("/")
    context = num_ctx if num_ctx is not None else config.get("NUM_CTX", OLLAMA_NUM_CTX)
    if not model or not url.startswith(("http://", "https://")) or type(context) is not int or context <= MAX_NEW_TOKENS:
        raise ValueError("팀의 모델 이름·Ollama 주소·문맥 한도를 확인하세요.")
    with generation_lock:
        if output_model is not None and (getattr(output_model, "model", None) != model
                or str(getattr(output_model, "base_url", "")).rstrip("/") != url
                or getattr(output_model, "num_ctx", None) != context):
            raise ValueError("이미 준비한 출력 연결과 팀 설정이 다릅니다. 새 런타임에서 연결하세요.")
        MODEL_NAME, OLLAMA_BASE_URL, OLLAMA_NUM_CTX = model, url, context
        input_namespace = {"search_rag": search_rag, "handoff_builder": handoff_builder,
            "scenario_cards": scenario_cards, "can_search": can_search,
            "memory_embedding_model": None, "input_version": "team"}
    return get_handoff


def get_handoff(player_input, state, npc_name="", history=None):
    """대화 대상이 있는 검색 질의와 실제 플레이어 발언을 분리합니다.

    검색은 팀 I2·I3를 한 번씩 그대로 사용합니다. 짧은 질문도 현재 누구에게
    한 말인지 검색기가 알 수 있게 이름만 덧붙이며 검색 원문·순위는 바꾸지 않습니다.
    """
    if input_namespace is None:
        raise RuntimeError("bind_team_backend로 팀 입력 함수를 먼저 연결하세요.")
    if not isinstance(state, dict):
        raise TypeError("게임 상태는 dict여야 합니다.")
    if not isinstance(player_input, str) or not player_input.strip():
        raise ValueError("플레이어 발언이 필요합니다.")
    target = canonical_npc_name(npc_name)
    # 이름이 질문에 이미 있어도 대화라는 맥락을 유지합니다. 팀 분류기의 인물 목록에
    # 없는 NPC도 동일한 경로를 쓰며, 규칙 단어가 있으면 원본 분류기가 BOTH로 처리합니다.
    retrieval_query = f"대화 상대 {target}: {player_input}" if target else player_input
    started = time.perf_counter()
    try:
        raw = input_namespace["search_rag"](query=retrieval_query, state=deepcopy(state), k=3)
        searched = time.perf_counter()
        original = input_namespace["handoff_builder"](search_result=raw, game_state=deepcopy(state))
    except Exception as error:
        # 검색 단계에서 실패해도 어떤 원문과 질의로 요청했는지 진단에 남깁니다.
        # 생성되지 않은 모델 메시지나 측정값은 만들어 채우지 않습니다.
        error.reply_to, error.retrieval_query = player_input, retrieval_query
        raise
    built = time.perf_counter()
    payload = read_handoff(original)
    payload["upstream_handoff"] = deepcopy(original)
    payload["player_input"] = player_input  # 검색용 문구를 실제 발언으로 저장하지 않습니다.
    payload["input_trace"] = {"original_player_input": player_input,
        "retrieval_query": retrieval_query, "intent": raw.get("intent"),
        "translated_queries": deepcopy(raw.get("translated_queries", []))}
    # 엔진의 추가 상태는 원본 handoff와 별도 필드로 둡니다.
    payload["engine_snapshot"] = deepcopy(state)
    payload["input_metrics"] = {"search_seconds": searched - started,
        "handoff_seconds": built - searched, "total_seconds": built - started}
    return payload


## 🟪 O3. 대화 기억·프롬프트

실제 최근 대화와 공통 생성 지침을 구성합니다. 주요 프롬프트는 이 셀에서 확인합니다.


In [ ]:
# O3. 접속자별 대화와 프롬프트를 구성합니다.
# 아래 지침·최근 대화·팀 근거·확정 게임 상태가 실제 모델 입력이 됩니다.
_bae_load_output_support("context_helpers.py")

_dialogue_sessions = {}
_dialogue_lock = threading.RLock()


def create_memory_session():
    """이번 게임의 대화를 보관할 빈 공간을 만듭니다."""
    token = secrets.token_urlsafe(24)
    with _dialogue_lock:
        _dialogue_sessions[token] = {"histories": {}, "turn": 0}
    return token


def load_dialogue_history(session_id, npc):
    """이번 게임에서 현재 상대와 주고받은 말을 복사해서 가져옵니다."""
    with _dialogue_lock:
        return deepcopy(_dialogue_sessions.get(session_id, {}).get("histories", {}).get(npc, []))


def save_dialogue_history(session_id, npc, history):
    """누가 한 말인지 함께 저장하고 최근 기록 100개를 남깁니다."""
    with _dialogue_lock:
        session = _dialogue_sessions.setdefault(session_id, {"histories": {}, "turn": 0})
        session["histories"][npc] = normalize_history(history)[-100:]


def close_memory_session(session_id):
    """해당 게임의 대화 메모만 지웁니다. 다른 게임의 메모는 유지합니다."""
    with _dialogue_lock:
        _dialogue_sessions.pop(session_id, None)


OUTPUT_SYSTEM = """당신은 한국어 TRPG의 현재 한 차례를 자연스러운 대사와 장면으로 작성한다.
참고 자료는 대본이 아니다. 설정 사실·관계는 의미와 주체를 유지하고, 성격·태도·말투는 인물을 연기하는 기준으로 사용한다. 인용 예시는 표현 참고이며 지금 실제로 한 말이 아니다. 제안·조건은 아직 실행되지 않은 일이고, 소문·추측은 그 확실성 그대로 표현한다.
실행된 행동·위치·소지품·피해·판정은 이번 엔진 기록을 따른다. 공개한 카드의 지식은 설정 근거이며 지급·이동을 실행한 기록이 아니다. 완료된 변화는 확인된 결과만 표현하고, 의향·제안을 새 보상이나 지급 완료로 확정하지 않는다.
NPC는 지정된 현재 인물의 1인칭과 제공된 한국어 말투로 직접 말한다. 상대의 최신 말에 먼저 답하고, 이번 목적에 필요한 안내만 짧게 덧붙인다. 자료의 역할명·식별자는 실제 호칭이 아니며, 상대의 이름을 모르면 이름을 붙이지 않고 말한다.
GM은 지금 일어난 결과와 눈앞의 장면 자체를 서술한다. 참고 기록이나 원문을 인용·해설하는 보고서 대신 완성된 이야기로 쓴다. 이전 장면을 현재 사건으로 되풀이하지 않는다.
NPC의 직접 대사는 speech, GM의 서술은 narration으로 나누고 지정된 blocks JSON만 반환한다."""
NPC_CONVERSATION_SYSTEM = """당신은 지정된 현재 NPC 본인이 되어 한국어로 한 차례 대화한다. 상대의 최신 질문·요청에 먼저 한두 문장으로 직접 답하고 필요한 진행 안내만 짧게 덧붙인다.
대사의 '나·제'는 현재 NPC이고 user 발언의 '나·제'는 상대다. 역할명·식별자는 실제 호칭이 아니며 이름을 모르면 이름을 붙이지 않고 말한다. 상대의 행동·감정·경험을 자신의 일로 옮기거나 미실행 행동·대사를 대신 정하지 않는다.
원문의 사실·관계는 의미와 주체를 유지한다. 성격·태도·말투 설명은 연기 기준으로 사용하고 대사로 읽지 않는다. 인용 예시는 현재 발언이 아니며 제안·조건은 완료가 아니다. 직접 발언 이력은 대화 흐름이고, 설정은 팀 원문을, 실제 완료·소지품·수치·위치는 엔진의 확정 자료를 따른다.
공개된 요청 조건은 읽기 전용이다. 대화로 새 보상·지급·이동을 확정하지 않고, 의향이나 협의와 실제 완료를 구별한다.
제공된 한국어 말투의 높임말과 종결형을 응답 전체에 일관되게 적용하고 지정된 현재 NPC의 speech 블록 JSON만 반환한다."""
NPC_CONVERSATION_FIRST_PASS = """인사·감사·감정·농담·잡담이나 대화 자체에 관한 말에는 현재 대화로 자연스럽게 답하고 mode=reply와 답변 blocks를 반환한다.
세계관·인물 관계나 이력·사건·장소·물품·규칙의 사실을 묻는 질문에는 mode=consult_reference, blocks=[]로 원문을 확인한다. 이전 대사만으로 사실 답변을 확정하거나 전체 목록이 완전하다고 가정하지 않는다.
JSON 형식은 {"mode":"reply 또는 consult_reference","blocks":[...]}이며 부가 설명을 붙이지 않는다."""
NPC_CONVERSATION_REFERENCE_PASS = """제공된 팀 원문에서 현재 질문에 필요한 사실을 확인해 현재 NPC의 말로 자연스럽게 답한다. 자료 문장을 대본처럼 읽지 않고 시간 범위와 소문·추측·확정의 차이를 유지한다. 필요한 사실이 없으면 모르는 범위만 짧게 말한다. 진행 안내는 현재 대화와 직접 연결될 때만 짧게 권한다.
이번에는 {"blocks":[...]} JSON만 반환한다. mode나 추가 자료 요청은 반환하지 않는다."""


def build_context(payload):
    """검색 원문·순서를 보존하고 표시용 자료 번호만 붙입니다."""
    sources = []
    for prefix, key in (("R", "rule_results"), ("S", "scenario_results")):
        for index, row in enumerate(payload["retrieval"].get(key, []), 1):
            meta = deepcopy(row.get("metadata") or {})
            sources.append({"label": prefix + str(index), "text": row["text"],
                "title": meta.get("title") or meta.get("card_id") or row.get("card_id") or key,
                "page": meta.get("page"), "score": row.get("score"), "metadata": meta})
    return "\n\n".join(row["text"] for row in sources), sources


def build_messages(payload, history=None, npc_name=None, engine_result=None):
    """팀 근거·현재 상태·최근 발언·이번 질문을 각자의 용도로 구분합니다.

    반환값은 Ollama에 보낼 메시지와 진단 표시용 출처 목록입니다.
    검색 문장을 재작성하거나 AI의 이전 대사에서 인물 설정을 추출하지 않습니다.
    """
    speaker = output_speaker(payload, npc_name)
    if npc_name is not None and payload.get("npc_name") != speaker:
        payload = _with_speaker_whitelist(payload, npc_name)
    _, sources = build_context(payload)
    contract = _utterance_task(payload)
    npc_mode = contract["mode"] == "npc_dialogue"
    if (contract["phase"] == "action_result" and contract["mode"] != "rule_explanation"
            and isinstance(engine_result, dict) and engine_result.get("confirmed") is True):
        return _build_current_action_messages(payload, speaker, contract, engine_result)
    state = payload.get("engine_snapshot") or payload.get("model_state") or payload["game_state"]
    if not isinstance(state, dict):
        raise ValueError("현재 모델용 상태는 dict여야 합니다.")
    # 검색 조건용 내부 vars·flags는 모델의 관찰 사실이 아닙니다.
    public_keys = {"scene", "cls", "current_room", "inventory", "gold_cp", "money", "hero", "combat",
        "플레이어 이름", "현재 장소", "플레이어 소지금", "플레이어 장착 장비", "플레이어가 보유한 소지품"}
    public_state = {key: value for key, value in state.items() if key in public_keys}
    public_state = _engine_data(public_state)
    snapshot = payload.get("player_snapshot")
    if isinstance(snapshot, dict):
        # 하나의 현재 플레이어 상태로 합쳐 가방·금액·HP를 중복 제시하지 않습니다.
        player = public_state.pop("hero", {})
        player = player if isinstance(player, dict) else {}
        for key in ("inventory", "gold_cp", "money"):
            if key in public_state:
                player[key] = public_state.pop(key)
        player.update(_engine_data(snapshot))
        public_state["player"] = player
    if npc_mode:
        public_state.pop("플레이어 이름", None)
        for key in ("player", "hero"):
            if isinstance(public_state.get(key), dict):
                public_state[key].pop("name", None)
                public_state[key].pop("id", None)
    # 참조 ID는 모델 입력의 사본에만 붙입니다. 원본 I3 문장·순서·출처는 유지합니다.
    referenced_sources = {}
    for prefix, key, group in (("R", "rule_results", "rules"), ("S", "scenario_results", "scenarios")):
        referenced_sources[group] = [{**deepcopy(row), "source_id": prefix + str(index)}
            for index, row in enumerate(payload["retrieval"].get(key, []), 1)
            if prefix + str(index) in contract["context_source_ids"]]
    data = {"현재 NPC": speaker or None,
        "팀 RAG 검색 결과": referenced_sources,
        "현재 엔진 상태": public_state,
        "읽기 전용 상호작용 계약": _current_interaction_contract(payload)}
    data.update(_speaker_context(payload))
    if npc_mode:
        data["dialogue_state"] = _npc_view(data["dialogue_state"])
    # 실행 ID 선택은 입력·추천 단계의 일입니다. 본문에는 공개 장면·진행 맥락을 줍니다.
    for key in ("scene_context", "story_progression", "progression_options", "participant_roles",
                "action_subject", "executed_action_ids", "engine_delta", "item_handovers"):
        if key in payload:
            value = _engine_data(payload[key])
            data[key] = _npc_view(value) if npc_mode else value
    if "dialogue_brief" in payload:
        data["dialogue_brief"] = deepcopy(payload["dialogue_brief"])
    data["진행 안내"] = _story_context(payload)
    if "instructions" in payload:
        data["팀 입력 지침"] = deepcopy(payload["instructions"])
    if engine_result is not None:
        # 대화 중에는 마지막 행동을 새로 실행한 것처럼 표현하지 않습니다.
        phase = (payload.get("dialogue_state") or {}).get("phase")
        key = "직전에 확정된 행동 결과" if phase == "continue" else "이번에 확정된 행동 결과"
        confirmed = _engine_data(validate_engine_result(engine_result))
        if payload.get("engine_snapshot"):
            confirmed.pop("state_after", None)  # 현재 상태는 위에 한 번 제시합니다.
        data[key] = _npc_view(confirmed) if npc_mode else confirmed
    if npc_mode:
        # NPC 대사 작성에는 검색 점수·전체 화자 목록·행동 라우팅 구조가 필요하지
        # 않습니다. 원문과 확정 사건은 유지하고 모델에게 보여 주는 사본만 줄입니다.
        # 원본 I3·엔진·전체 진단 기록은 여기서 수정하지 않습니다.
        state_for_dialogue = {key: deepcopy(public_state[key]) for key in
            ("scene", "current_room", "현재 장소", "inventory", "gold_cp", "money",
             "플레이어 소지금", "플레이어가 보유한 소지품") if key in public_state}
        if isinstance(public_state.get("player"), dict):
            state_for_dialogue["player"] = {key: deepcopy(public_state["player"][key]) for key in
                ("inventory", "gold_cp", "money") if key in public_state["player"]}
        data = {"이번 화자": deepcopy(contract["speaker"]),
            "화자의 인적 사항 원문 번호": contract["speaker_source_ids"],
            "말투": _speech_style(contract["speaker"]["speaker_id"]),
            "팀 RAG 검색 결과": {group: [{"source_id": row["source_id"], "text": row["text"]}
                for row in rows] for group, rows in referenced_sources.items()},
            "현재 엔진 상태": state_for_dialogue,
            "현재 공개 상호작용": _npc_view(_engine_data(payload.get("interaction_context") or {})),
            "읽기 전용 상호작용 계약": _current_interaction_contract(payload),
            "진행 안내": _story_context(payload),
            **{key: value for key, value in data.items() if key in
                ("이번에 확정된 행동 결과", "직전에 확정된 행동 결과")}}
    start = {"role": "system", "content": OUTPUT_SYSTEM}
    context = {"role": "system", "content": "이번 응답의 참고 자료입니다. 등장인물의 발언이 아닙니다.\n" + _json_text(data)}
    turn = {"role": "system", "content": "이번 턴 계약\n" + contract["task"] + "\n" + _json_text(contract)}
    current = {"role": "user", "content": payload["player_input"]}
    recent = _recent_messages(history, speaker_id=contract["speaker"]["speaker_id"])
    if contract["mode"] == "scene_dialogue":
        recent = []  # 새 장면의 사실을 이전 생성 문장에서 가져오지 않습니다.
    messages = [start, *recent, context, turn, current]
    # 원본 검색 결과·현재 질문을 유지하고 오래된 대화만 줄입니다.
    while recent and _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        recent.pop(0)
        messages = [start, *recent, context, turn, current]
    # 검색 전체 원본은 payload에 남고, 이 목록은 실제 모델에 제시한 자료입니다.
    return messages, [row for row in sources if row["label"] in contract["context_source_ids"]]


def build_npc_conversation_messages(payload, history, speaker, include_reference=False, engine_result=None):
    """대화를 먼저 이어가고 꼭 필요한 사실만 원문에서 확인하도록 구성합니다.

    잡담 요청에는 화자·말투와 실제 대화만 줍니다. 사실 확인이 필요한 요청은
    이미 받은 팀 I3 원문과 현재 공개 상태를 함께 제공합니다. 원본 검색·상태·
    이력은 바꾸지 않으며 인물 설정을 추출하거나 별도 사실 캐시를 만들지 않습니다.
    """
    current_speaker = output_speaker(payload, speaker)
    local = payload if payload.get("npc_name") == current_speaker else _with_speaker_whitelist(payload, current_speaker)
    contract = _utterance_task(local)
    if contract["mode"] != "npc_dialogue":
        raise ValueError("자유 대화 구성에는 현재 NPC 대화 모드가 필요합니다.")
    identifier = contract["speaker"]["speaker_id"]
    style = _speech_style(identifier)
    data = {"현재 화자": deepcopy(contract["speaker"]), "말투": style,
        "발언의 주체": {"마지막 user 발언": "상대", "이번 assistant 응답": current_speaker},
        "읽기 전용 상호작용 계약": _current_interaction_contract(local),
        "이번 입력": {"phase": contract["phase"], "input_kind": contract["input_kind"]},
        "출력 계약": deepcopy(contract["output"])}
    actual_sources = []
    if include_reference:
        _, actual_sources = build_context(local)
        full_references = {"rules": [], "scenarios": []}
        for source in actual_sources:
            group = "rules" if source["label"].startswith("R") else "scenarios"
            full_references[group].append({"source_id": source["label"], "text": source["text"]})
        scene = local.get("scene_context") or {}
        data["팀 RAG 검색 결과"] = full_references
        data["현재 장소"] = {key: deepcopy(scene[key]) for key in
            ("room_id", "room_name", "observation", "people") if key in scene}
        data["현재 공개 상호작용"] = _npc_view(_engine_data(local.get("interaction_context") or {}))
        data["진행 안내"] = _story_context(local)
        # 이전 행동의 전체 장면 문장은 재사용하지 않고 실제 전달 영수증만 제공합니다.
        if isinstance(engine_result, dict):
            data["확인된 물품 전달"] = _npc_view(_engine_data(engine_result.get("item_handovers") or []))
    instruction = NPC_CONVERSATION_REFERENCE_PASS if include_reference else NPC_CONVERSATION_FIRST_PASS
    start = {"role": "system", "content": NPC_CONVERSATION_SYSTEM + "\n" + instruction}
    context = {"role": "system", "content": "현재 대화의 참고 자료입니다. 대화 이력은 실제 발언이며 사실의 증명은 아닙니다.\n" + _json_text(data)}
    # 참고 자료를 먼저 주고 실제 대화를 이어 붙여 가장 최근 발언을 마지막에 둡니다.
    recent = _plain_npc_conversation(history, identifier)
    current = {"role": "user", "content": local["player_input"]}
    messages = [start, context, *recent, current]
    while recent and _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        recent.pop(0)
        messages = [start, context, *recent, current]
    return messages, actual_sources


## 🟪 O4. Qwen 호출·답변 작성

Ollama에 요청하고 NPC 대사·확정 행동 결과를 작성합니다. 반복 검사와 스트리밍은 보조 코드가 처리합니다.


In [ ]:
# O4. Ollama에 생성 요청을 보내고 NPC 대사와 행동 결과를 만듭니다.
# 스트리밍·형식 검사·오류 정리는 보조 함수가 담당합니다.
_bae_load_output_support("response_validation.py")
_bae_load_output_support("generation_helpers.py")

def load_output_model(pull=False):
    """설치된 Ollama 모델을 확인합니다. pull=True일 때만 내려받습니다."""
    global output_model
    from ollama_backend import OllamaBackend
    with generation_lock:
        if output_model is None:
            backend = OllamaBackend(MODEL_NAME, OLLAMA_BASE_URL,
                keep_alive=OLLAMA_KEEP_ALIVE, num_ctx=OLLAMA_NUM_CTX)
            backend.prepare(pull=pull)
            output_model = backend
    return output_model


def run_generation(messages, max_new_tokens=MAX_NEW_TOKENS, sample=None, thinking=False,
                   temperature=None, top_p=None, top_k=None, on_text=None, cancel_event=None,
                   response_format=None):
    """한 요청을 실행하고 실제 토큰·시간·적용 옵션을 함께 반환합니다.

    sample이 없으면 공통 출력 설정을 따릅니다. 실패 시에도 시도한 옵션을
    예외에 붙여 다음 진단 단계가 추정값을 기록하지 않도록 합니다.
    """
    # 블록이 완성되기 전에도 UI 종료·추천 취소를 같은 경로에서 확인합니다.
    inherited_events = _output_stream_cancel_events.get()
    check_cancel = lambda: _raise_if_cancelled(cancel_event, inherited_events=inherited_events)
    check_cancel()
    if output_model is None:
        raise RuntimeError("출력 모델을 먼저 준비하세요.")
    if _prompt_characters(messages) > MAX_INPUT_CHARACTERS:
        raise ValueError("전달 자료가 너무 깁니다. 팀 검색 개수나 입력 분량을 확인하세요.")
    if type(max_new_tokens) is not int or not 0 < max_new_tokens < OLLAMA_NUM_CTX:
        raise ValueError("생성 한도는 Ollama 문맥 한도보다 작은 양의 정수여야 합니다.")
    format_options = {"response_format": response_format} if response_format is not None else {}
    sampled = DO_SAMPLE if sample is None else sample
    options = {"num_ctx": OLLAMA_NUM_CTX, "num_predict": max_new_tokens,
        "sample": sampled, "temperature": (TEMPERATURE if temperature is None else temperature) if sampled else 0,
        "top_p": (TOP_P if top_p is None else top_p) if sampled else None,
        "top_k": (TOP_K if top_k is None else top_k) if sampled else None,
        "thinking": thinking, "structured_response": response_format is not None}
    try:
        with _generation_slot(cancel_event):
            result = output_model.generate(messages, max_new_tokens=max_new_tokens,
                sample=sampled, thinking=thinking,
                temperature=TEMPERATURE if temperature is None else temperature,
                top_p=TOP_P if top_p is None else top_p, top_k=TOP_K if top_k is None else top_k,
                on_text=on_text, check_cancel=check_cancel, **format_options)
            check_cancel()
    except BaseException as error:
        error.request_options = deepcopy(options)
        raise
    result["generation_limit"] = max_new_tokens
    result["request_options"] = options
    return result


def generate_from_handoff(value, history=None, npc_name=None, engine_result=None):
    """원본 I3 JSON을 직접 받아 검색 결과와 최근 대화로 본문을 생성합니다."""
    payload = _with_speaker_whitelist(read_handoff(value), npc_name)
    speaker = output_speaker(payload, npc_name)
    engine_result = engine_result if engine_result is not None else payload.get("engine_result")
    if isinstance(engine_result, dict) and (payload.get("dialogue_state") or {}).get("phase") == "action_result":
        payload["engine_result"] = validate_engine_result(engine_result)
    payload = _with_action_speaker_scope(payload, engine_result)
    messages, sources = build_messages(payload, history, speaker, engine_result)
    prepared = _prepared_opening(payload, speaker, sources, engine_result=engine_result)
    if prepared is not None:
        return prepared
    outcome = _prepared_confirmed_interaction(payload, speaker, sources, engine_result=engine_result)
    if outcome is not None:
        return outcome
    task = _utterance_task(payload)
    if task["mode"] == "npc_dialogue" and task["phase"] == "continue" and task["input_kind"] == "speech":
        return _generate_npc_conversation(payload, history, speaker, engine_result=engine_result)
    return _generate_main_result(messages, payload, speaker, sources, engine_result)


def generate_action_result(payload, *, handoff=None, engine_result=None):
    """팀 I3 근거와 별도 확정 사건으로 행동 뒤의 표현을 생성합니다.

    handoff 또는 payload.input_handoff에 이미 검색한 팀 JSON을 전달합니다.
    출력은 재검색하지 않습니다. handoff 없는 호출은 엔진 결과만으로
    처리하며 public_narratives의 직접 카드 본문은 생성 입력에 넣지 않습니다.
    """
    if not isinstance(payload, dict):
        raise TypeError("행동 결과는 dict여야 합니다.")
    caller = deepcopy(payload)
    confirmed = validate_engine_result(engine_result if engine_result is not None else caller.get("engine_result"))
    original = handoff if handoff is not None else caller.get("input_handoff")
    if original is None and ("retrieval" in caller or "retrieved_context" in caller):
        original = caller
    if original is not None:
        prepared = read_handoff(original)
        evidence_status = "team_handoff"
    else:
        state = caller.get("engine_snapshot") or confirmed.get("state_after") or caller.get("game_state") or {}
        prepared = read_handoff({"player_input": caller.get("player_input"), "game_state": _engine_data(state),
            "retrieval": {"rule_results": [], "scenario_results": [], "memory_results": []}})
        evidence_status = "engine_only_no_input_handoff"
    # 실행 시점의 역할·수치만 별도로 연결합니다. 지식·카드 본문은 I3에서만 받습니다.
    for key in ("npc_name", "actor_context", "speaker_id", "allowed_speakers", "dialogue_state",
                "response_mode", "action_kind", "movement_scene", "interaction_context",
                "story_progression", "confirmed_offer", "needs_information", "reference_required", "turn_intent",
                "scene_context", "player_snapshot", "engine_snapshot", "model_state", "available_actions",
                "progression_options", "participant_roles", "action_subject", "executed_action_ids",
                "engine_delta", "item_handovers"):
        if key in caller:
            prepared[key] = _engine_data(caller[key])
    phase = deepcopy(prepared.get("dialogue_state") or {})
    if not isinstance(phase, dict):
        raise ValueError("dialogue_state는 객체여야 합니다.")
    phase.setdefault("phase", "action_result")
    prepared["dialogue_state"] = phase
    prepared["engine_result"] = confirmed
    prepared = _with_speaker_whitelist(prepared)
    prepared = _with_action_speaker_scope(prepared, confirmed)
    speaker = output_speaker(prepared)
    if not isinstance(prepared.get("player_input"), str) or not prepared["player_input"].strip():
        raise ValueError("실행한 행동의 문구가 필요합니다.")
    messages, sources = build_messages(prepared, caller.get("recent_conversation"), speaker, confirmed)
    opening = _prepared_opening(prepared, speaker, sources, engine_result=confirmed)
    if opening is not None:
        opening["evidence_status"] = evidence_status
        return opening
    outcome = _prepared_confirmed_interaction(prepared, speaker, sources, engine_result=confirmed)
    if outcome is not None:
        outcome["evidence_status"] = evidence_status
        return outcome
    result = _generate_main_result(messages, prepared, speaker, sources, confirmed)
    result["evidence_status"] = evidence_status
    if evidence_status != "team_handoff":
        result["warnings"].append("팀 입력 근거가 없어 엔진 확정 결과만 사용했습니다.")
    return result


## 🟪 O5. 다음 선택지

방금 대화와 실제 가능한 행동으로 다음 선택지를 만듭니다. 추천은 게임 상태를 바꾸지 않습니다.


In [ ]:
# O5. 방금 대화와 현재 가능한 행동으로 후속 선택지를 추천합니다.
# 추천은 문장과 엔진 행동 ID를 연결하며 게임 상태를 변경하지 않습니다.
def generate_followup_question(payload, reply, history=None, cancel_event=None):
    """방금 대화와 실제 가능한 행동으로 다음 선택지를 만듭니다.

    대화 추천은 입력 문장이고, 행동 추천은 엔진 ID에 연결됩니다.
    추천 생성은 세계 상태나 소지품을 바꾸지 않습니다.
    """
    _raise_if_cancelled(cancel_event)
    count = max(1, min(3, int(payload.get("suggestion_count", SUGGESTION_COUNT))))
    actions = deepcopy(payload.get("available_actions", []))
    data = {"현재 NPC": output_speaker(payload), "플레이어 원문": payload.get("player_input"),
        "방금 본문": reply, "최근 대화": _recent_messages(history)[-6:], "실제 가능한 행동": actions}
    messages = [{"role": "system", "content":
        f'이번 대화에 이어 플레이어가 다음에 할 말이나 행동을 {count}개 추천한다.\n'
        '현재 질문의 목적을 이어가는 짧은 선택지를 만들고, 플레이어가 말하지 않은 목격·경험을 지어내지 않는다.\n'
        '대화는 플레이어가 할 말이고 행동·이동은 제공된 현재 행동 ID와 그 이름을 그대로 사용한다.\n'
        'JSON 계약: {"options":[{"kind":"dialogue|action|move","text":"플레이어의 말 또는 행동","action_id":"연결할 때의 실제 ID"}]}.'},
        {"role": "user", "content": _json_text(data)}]
    generated = run_generation(messages, max_new_tokens=160, sample=False, cancel_event=cancel_event)
    _raise_if_cancelled(cancel_event)
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", generated["text"].strip()).strip()
    try:
        value = json.loads(text)
    except ValueError:
        value = {}
    allowed = {row["id"]: row for row in actions if isinstance(row, dict) and isinstance(row.get("id"), str)
               and isinstance(row.get("label"), str) and row["label"].strip()}
    options = []
    from response_guard import inspect_response
    rows = value.get("options", []) if isinstance(value, dict) else []
    for row in rows if isinstance(rows, list) else []:
        if not isinstance(row, dict) or row.get("kind") not in {"dialogue", "action", "move"}:
            continue
        label = row.get("text")
        if (not isinstance(label, str) or not 1 <= len(label.strip()) <= 140
                or inspect_response(label)["hard_issues"]):
            continue
        identifier = row.get("action_id")
        if identifier and (not isinstance(identifier, str) or identifier not in allowed):
            continue
        if row["kind"] != "dialogue" and not identifier:
            continue  # 실행할 수 있는 행동만 추천합니다.
        if row["kind"] == "dialogue" and identifier:
            continue  # 대화 문장을 누른 뒤 다른 행동이 실행되지 않게 합니다.
        option = {"kind": row["kind"], "text": label.strip()}
        if identifier:
            option["action_id"] = identifier
            # 화면의 행동 이름과 클릭 후 실행되는 행동이 서로 달라지지 않게 합니다.
            option["text"] = allowed[identifier]["label"]
            option["kind"] = "move" if identifier.startswith("move:") else "action"
        if option not in options:
            options.append(option)
        if len(options) >= count:
            break
    # 선택지가 없는 원인을 구분해 진단에 남깁니다. 임의의 고정 답변으로 채우지 않습니다.
    truncated = generated.get("done_reason") == "length" or generated.get("output_tokens", 0) >= generated.get("generation_limit", 160)
    if options:
        reason = "generated"
    elif truncated:
        reason = "generation_limit"
    elif not isinstance(value, dict) or not isinstance(value.get("options"), list):
        reason = "invalid_json"
    else:
        reason = "no_valid_options"
    return {"text": options[0]["text"] if options else "", "suggestions": [row["text"] for row in options],
        "recommendation_options": options, "seconds": generated["seconds"],
        "input_tokens": generated["input_tokens"], "output_tokens": generated["output_tokens"], "reason": reason}


## 🟪 O6. 게임 화면 열기

출력 함수와 팀 함수를 연결해 Gradio 게임 화면을 엽니다. 실제 GPU·모델·양자화도 표시합니다.


In [ ]:
# O6. 위에서 정의한 출력 함수와 팀 함수를 연결해 Gradio 게임을 엽니다.
# 화면 구성·파일 버전 확인은 읽을 수 있는 보조 코드에 모았습니다.
_bae_load_output_support("gui_bridge.py")

bae_demo = _bae_connect_and_open(globals())


## 🟪 실행

팀 공통·룰·시나리오·입력 준비 후 O1~O6을 실행합니다. GitHub에서 받을 보조 코드는 `BAE/src`에 공개되어 있습니다. 파일 선택 창이나 모델 중복 설치는 없습니다.
